# Lecture 8.8 — Dynamic State Management with `after_tool_callback`

**Design Pattern:** Dynamic State Management (P2) — also reinforces Logging & Monitoring (P3) from 8.3
**Callbacks used:** `after_tool_callback` (on `sum_costs`) + `before_agent_callback` (on `plan_retriever_agent`)
**Applied to:** `accountant_agent` (write side) → `plan_retriever_agent` (read side)

In this lecture we implement a **state bridge** between two callbacks that never call each other — they communicate exclusively through session state.

- `after_tool_callback` on `accountant_agent` → **writes**: extracts the running total from every `sum_costs` result and appends it to `state['budget_history']`
- `before_agent_callback` on `plan_retriever_agent` → **reads**: consumes `state['budget_history']` and writes a human-readable summary into `state['refinement_summary']`

The two callbacks communicate through state — the entire point of the Dynamic State Management pattern.

---
**Changes from Lecture 8.7 (the complete diff):**
1. New function: `track_budget_history(tool, args, tool_context, tool_response)` — the `after_tool_callback` on `accountant_agent` (write side)
2. New function: `enrich_final_output(callback_context)` — the `before_agent_callback` on `plan_retriever_agent` (read side)
3. `accountant_agent` gains `after_tool_callback=track_budget_history` (joins the existing `before_tool_callback=audit_and_validate_sum_costs`)
4. `plan_retriever_agent` gains `before_agent_callback=enrich_final_output` (joins the existing `after_agent_callback=log_agent_exit`)
5. `plan_retriever_agent` instruction updated to include `{{refinement_summary}}` template variable — the only instruction change in the entire Section 8
6. New state variables introduced: `state['budget_history']` (List[float]) and `state['refinement_summary']` (str)
7. New standalone demo cell — Scenarios 1 and 2 with mock objects
8. Capstone full workflow execution cell — all nine callbacks firing simultaneously

---
**State schema introduced:**
```python
state['budget_history']      # List[float] — one entry per loop iteration
                             # e.g. [21000.0, 16500.0, 12400.0]

state['refinement_summary']  # str — human-readable summary written by enrich_final_output
                             # e.g. "Refined across 3 iterations. Total reduced from
                             #        $21,000 → $12,400 (saving $8,600)."
```

---
**Critical insight — list accumulation in state:**

```python
# ✅ CORRECT: read → append → reassign
history = tool_context.state.get('budget_history', [])
history.append(total)
tool_context.state['budget_history'] = history

# ❌ WRONG: in-place mutation on a nested object
tool_context.state['budget_history'].append(total)   # state proxy may not propagate this
```

---
**Expected output — budget_history accumulating across iterations:**
```
=== Loop Iteration 1 ===
[AUDIT]   sum_costs called | costs: [10000.0, 8000.0]
[HISTORY] Appended 18000.0 to budget_history → [18000.0]

=== Loop Iteration 2 ===
[AUDIT]   sum_costs called | costs: [7500.0, 8000.0]
[HISTORY] Appended 15500.0 to budget_history → [18000.0, 15500.0]
```
**Expected output — enrichment firing before plan_retriever_agent:**
```
[ENRICH] plan_retriever_agent starting
[ENRICH] budget_history found: [18000.0, 15500.0, 12400.0]
[ENRICH] Wrote refinement_summary to state:
         "Refined across 3 iterations. Total reduced from $18,000 → $12,400 (saving $5,600)."
[ENRICH] Returning None — agent proceeds normally with enriched state.
```


## ⚙️ 1. Setup: Install Libraries

Pinning the version ensures our code will always work as expected.

In [ ]:
!pip install google-adk==1.29.0 -q

## 🔑 2. Authentication: Configure Your API Key

In [ ]:
import os
from getpass import getpass

api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully!")

Enter your Google API Key: ··········
✅ API Key configured successfully!


## 🤖 3. Model Configuration

Define the model names once here. To upgrade to a newer model in the future,
change these two constants — nothing else in the notebook needs to touch.

In [ ]:
# ── Model Configuration ───────────────────────────────────────────────────────
# Change these two constants to swap models across the entire notebook.
# To upgrade to a newer model in the future, update AGENT_MODEL and JUDGE_MODEL here.

AGENT_MODEL = "gemini-2.5-flash"  # used by all workflow agents
JUDGE_MODEL = "gemini-2.5-flash"   # used by the safety judge

## 🪝 4. [CARRIED OVER from 8.3, 8.4, 8.5, 8.6 & 8.7] Define the Observability Callbacks

These two callbacks are unchanged from Lecture 8.3 and carried forward through 8.4, 8.5, 8.6, and 8.7.
They fire at the **agent** boundary (entry and exit).

The new dynamic state callbacks in this lecture also fire at the **agent** and **tool** boundaries — but on different agents.

All nine callbacks now coexist and fire simultaneously during a live run:

| Callback | Hook | Agent | Pattern |
|---|---|---|---|
| `guardrail_before_workflow` | `before_agent_callback` | `budget_optimizer_workflow` | P1 Guardrails |
| `log_agent_entry` | `before_agent_callback` | `spending_proposer_agent` | P3 Logging |
| `sanitize_cost_cutter_response` | `after_model_callback` | `cost_cutter_agent` | P5 Response Modification |
| `audit_and_validate_sum_costs` | `before_tool_callback` | `accountant_agent` | P5 Tool-layer Modification |
| `track_budget_history` | `after_tool_callback` | `accountant_agent` | P2 Dynamic State ← **8.8 NEW** |
| `read_search_cache` | `before_tool_callback` | `cost_cutter_agent` | P4 Caching |
| `write_search_cache` | `after_tool_callback` | `cost_cutter_agent` | P4 Caching |
| `enrich_final_output` | `before_agent_callback` | `plan_retriever_agent` | P2 Dynamic State ← **8.8 NEW** |
| `log_agent_exit` | `after_agent_callback` | `plan_retriever_agent` | P3 Logging |


In [ ]:
from datetime import datetime
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types

# A module-level variable so log_agent_exit can calculate elapsed time.
_workflow_start_time: datetime = None


def log_agent_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    before_agent_callback for spending_proposer_agent.

    Fires once, right before the first LLM call in the entire workflow.
    Records the start time and prints a structured ENTRY log line.

    Returns None — the agent proceeds normally. Nothing is skipped.
    """
    global _workflow_start_time
    _workflow_start_time = datetime.now()  # Capture start time for later

    # --- Read from CallbackContext ---
    agent_name    = callback_context.agent_name      # e.g. 'spending_proposer_agent'
    invocation_id = callback_context.invocation_id   # unique UUID for this run
    state_keys    = list(callback_context.state.to_dict().keys())  # what's in memory so far
    timestamp     = _workflow_start_time.strftime("%H:%M:%S")

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[ENTRY] {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        state_keys : {state_keys}")
    print("="*60)

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — proceed with the agent as normal.'
    return None


def log_agent_exit(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    after_agent_callback for plan_retriever_agent.

    Fires once, right after the last agent in the workflow completes.
    Calculates total elapsed time and prints a structured EXIT log line.

    Returns None — the agent's output is used unchanged. Nothing is replaced.
    """
    now        = datetime.now()
    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    state_keys = list(callback_context.state.to_dict().keys())
    timestamp  = now.strftime("%H:%M:%S")

    # Calculate duration only if log_agent_entry ran first
    if _workflow_start_time is not None:
        elapsed = (now - _workflow_start_time).seconds
        duration_str = f"{elapsed}s"
    else:
        duration_str = "n/a"

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[EXIT]  {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        duration   : {duration_str}")
    print(f"        state_keys : {state_keys}")
    print("="*60 + "\n")

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — use the agent's real output as-is.'
    return None

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## 🛡️ 5. [CARRIED OVER from 8.4] Define the LLM-as-Judge Guardrail

This cell is **unchanged from Lecture 8.4**. The guardrail fires at the **agent** boundary on `budget_optimizer_workflow` — before any sub-agent starts.

### Why it is still here

The 8.7 caching callbacks operate at the **tool** boundary on `cost_cutter_agent` — a completely different level. All callbacks coexist without interference:

| Callback | Hook | Fires on | Boundary |
|---|---|---|---|
| `guardrail_before_workflow` | `before_agent_callback` | `budget_optimizer_workflow` | Agent |
| `log_agent_entry` | `before_agent_callback` | `spending_proposer_agent` | Agent |
| `sanitize_cost_cutter_response` | `after_model_callback` | `cost_cutter_agent` | Model |
| `audit_and_validate_sum_costs` | `before_tool_callback` | `accountant_agent` | Tool |
| `read_search_cache` | `before_tool_callback` | `cost_cutter_agent` | Tool |
| `write_search_cache` | `after_tool_callback` | `cost_cutter_agent` | Tool |
| `log_agent_exit` | `after_agent_callback` | `plan_retriever_agent` | Agent |

### Return value contract (reminder)
- Return `None` → topic is safe, workflow runs normally
- Return `Content` → entire workflow cancelled instantly — no sub-agent ever starts


In [ ]:
# ============================================================
# Lecture 8.4 — LLM-as-Judge Guardrail  [CARRIED OVER — UNCHANGED]
# Design Patterns: Guardrails & Policy Enforcement (P1)
#                  Conditional Skipping of Steps (P6)
# ============================================================

from google.genai import client as genai_client

# Initialise a direct Gemini client for the safety judge.
# This is a raw API call — completely separate from the ADK runner.
safety_client = genai_client.Client()

# ── Judge prompts ─────────────────────────────────────────────────────────
# Two-stage design:
#   Prompt 1 — binary verdict (YES/NO). Fast, cheap, used on every request.
#   Prompt 2 — human-readable explanation. Only called when verdict is YES,
#              so clean topics pay no extra cost.

SAFETY_VERDICT_PROMPT = """
You are a strict safety officer for an event planning company.
Evaluate the following event planning request.

Does it involve any of the following:
- Dangerous or high-risk activities
- Weapons, arms, or military equipment
- Illegal substances or narcotics
- Illegal activities of any kind
- Activities that cannot be commercially insured
- Adult-only or explicit content
- Anything that exposes the company to legal or reputational risk

Reply with EXACTLY one word — either YES or NO.
No explanation. No punctuation. Just the single word.

Event request: "{topic}"
"""

SAFETY_REASON_PROMPT = """
You are a polite but firm safety officer for an event planning company.
A client has requested help planning an event, but it has been flagged as
unsafe or inappropriate for our business.

Write a short, professional refusal message (2-3 sentences) addressed to
the client. Explain specifically why this type of event falls outside what
the company can assist with. Be clear but courteous. Do not offer
workarounds or alternatives.

Event request: "{topic}"
"""


def guardrail_before_workflow(
    callback_context: CallbackContext,
) -> Optional[types.Content]:
    """
    before_agent_callback on budget_optimizer_workflow (the SequentialAgent).

    Two-stage LLM-as-judge:
      Stage 1 — fast binary verdict (YES/NO) on every request.
      Stage 2 — rich refusal explanation, only when Stage 1 says YES.

    The explanation is written into state["refusal_reason"] so the runner
    can surface it as the final response instead of "No plan found."

    Placed on the SequentialAgent so it fires once before ANY sub-agent
    starts. Returning Content cancels the entire workflow instantly.
    """
    topic = callback_context.state.get("topic", "")

    print("\n" + "─" * 60)
    print(f"[SAFETY JUDGE] Evaluating topic: '{topic}'")

    # ── Stage 1: Binary verdict ───────────────────────────────────────────
    verdict_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_VERDICT_PROMPT.format(topic=topic),
    )
    verdict = verdict_response.text.strip().upper()
    print(f"[SAFETY JUDGE] Verdict         : {verdict}")

    if "YES" not in verdict:
        print(f"  └─ ✅ SAFE — starting workflow.")
        print("─" * 60)
        return None                        # topic is safe, proceed normally

    # ── Stage 2: Rich explanation (only reached when blocked) ─────────────
    print(f"[SAFETY JUDGE] Generating refusal explanation...")
    reason_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_REASON_PROMPT.format(topic=topic),
    )
    refusal_reason = reason_response.text.strip()
    print(f"[SAFETY JUDGE] Reason          : {refusal_reason}")
    print(f"  └─ 🚫 BLOCKED — high-risk topic. Workflow cancelled.")
    print("─" * 60)

    # Write the rich explanation into session state.
    # The runner reads state["refusal_reason"] when state["final_presentation"]
    # is absent — this is how the final response reaches the user.
    callback_context.state["refusal_reason"] = refusal_reason

    # Return Content to cancel the entire workflow.
    return types.Content(
        role="model",
        parts=[types.Part(text=refusal_reason)],
    )

## 🧹 6. [CARRIED OVER from 8.5] Define the Response Sanitization Callback

This cell is **unchanged from Lecture 8.5**. The `after_model_callback` on `cost_cutter_agent` strips markdown fences and coerces string costs to floats before the framework processes the LLM response.

It fires at the **model** boundary. The new 8.7 caching callbacks fire at the **tool** boundary — a later, narrower interception point that sees the final argument values the LLM produced for the tool call.

| Callback | What it fixes | When it fires |
|---|---|---|
| `sanitize_cost_cutter_response` | Fence-wrapped JSON, string costs in plan | After `cost_cutter_agent` LLM responds |
| `read_search_cache` | Returns cached result instead of calling tool | Before `google_search_tool` executes |
| `write_search_cache` | Stores fresh result for future hits | After `google_search_tool` executes |


In [ ]:
# ============================================================
# Lecture 8.5 — Response Sanitization: after_model_callback
# Design Pattern: Request / Response Modification (P5)
# ============================================================

import re
import json
from google.adk.models import LlmResponse
from google.genai import types as genai_types


def sanitize_cost_cutter_response(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> Optional[LlmResponse]:
    """
    after_model_callback for cost_cutter_agent.

    Intercepts the raw LLM response and fixes two consistent formatting flaws:
      1. Markdown code fences   -- ```json ... ```  wrapping the JSON
      2. String costs           -- "cost": "5000" instead of "cost": 5000

    Return value contract:
      - Function call response       -> return None immediately (don't touch tool calls)
      - Fences or string costs found -> return a NEW LlmResponse with cleaned content
      - Already clean               -> return None (original passes through unchanged)

    Why return a new object rather than mutating the original?
    Other callbacks in the chain may hold references to the original LlmResponse.
    Mutating it in-place would silently affect those other callbacks.
    Always build a new one.
    """
    agent_name = callback_context.agent_name

    # -- Guard: only process text responses ---------------------------------
    # LLM responses can contain function calls, not just text.
    # Check before assuming parts[0] is a text part.
    if (
        not llm_response.content
        or not llm_response.content.parts
        or llm_response.content.parts[0].function_call is not None
    ):
        return None   # Tool call — pass through untouched

    raw_text = llm_response.content.parts[0].text
    if not raw_text:
        return None

    cleaned = raw_text
    modified = False
    coerce_count = 0

    # -- Fix 1: Strip markdown code fences ----------------------------------
    fence_pattern = r"^\s*```(?:json)?\s*\n?(.*?)\n?\s*```\s*$"
    fence_match = re.search(fence_pattern, cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
        modified = True
        print(f"[SANITIZE] Stripped markdown fences from {agent_name} response")

    # -- Fix 2: Coerce string costs to float --------------------------------
    # Walk the parsed JSON and convert any value whose key contains 'cost'
    # from a string to a float.
    try:
        data = json.loads(cleaned)

        def coerce_costs(obj):
            nonlocal coerce_count
            if isinstance(obj, dict):
                for key, value in obj.items():
                    if "cost" in key.lower() and isinstance(value, str):
                        try:
                            obj[key] = float(value)
                            coerce_count += 1
                        except ValueError:
                            pass   # leave non-numeric strings alone
                    else:
                        coerce_costs(value)
            elif isinstance(obj, list):
                for item in obj:
                    coerce_costs(item)

        coerce_costs(data)

        if coerce_count > 0:
            cleaned = json.dumps(data)
            modified = True
            print(f"[SANITIZE] Coerced {coerce_count} string cost(s) to float in {agent_name} response")

    except (json.JSONDecodeError, TypeError):
        # Not valid JSON — skip coercion, but still return cleaned text
        # if fences were stripped
        pass

    # -- Return -------------------------------------------------------------
    if not modified:
        print(f"[SANITIZE] Response already clean. Passing through.")
        return None   # No change needed — return None so original is used

    # Build a NEW LlmResponse — never mutate the original
    cleaned_content = genai_types.Content(
        role=llm_response.content.role,
        parts=[genai_types.Part(text=cleaned)],
    )
    return LlmResponse(content=cleaned_content)

## 🔍 7. [CARRIED OVER from 8.6 & 8.7] High-Cost Threshold & Tool Auditing Callback

These two cells are **unchanged from Lecture 8.6**. The `HIGH_COST_THRESHOLD` constant and `audit_and_validate_sum_costs` are untouched.

In this lecture, `accountant_agent` gets a **second** tool callback alongside this one:
- `before_tool_callback=audit_and_validate_sum_costs` ← 8.6 (unchanged)
- `after_tool_callback=track_budget_history` ← **8.8 NEW** (write side of the state bridge)

Both callbacks fire on the same agent — `audit_and_validate_sum_costs` fires before `sum_costs` executes, `track_budget_history` fires after it returns. They do not interfere.


In [ ]:
# ── 8.6 Constant (unchanged) ──────────────────────────────────────────────────
# Costs above this value trigger a [FLAG] warning log line in the auditor.
# The callback still proceeds — this is observation-only, not blocking.

HIGH_COST_THRESHOLD = 10_000   # configurable — set low enough to flag luxury venue/catering costs in the workflow

In [ ]:
# ============================================================
# Lecture 8.6 — Tool Auditing and Argument Validation  [CARRIED OVER — UNCHANGED]
# Design Pattern: Request / Response Modification (P5) at the tool layer
# ============================================================

from typing import Any, Dict
from google.adk.tools import ToolContext
from google.adk.tools.base_tool import BaseTool

# Module-level iteration counter — tracks which loop pass triggered the tool call.
# Reset to 0 each time you run a fresh workflow.
_sum_costs_iteration: int = 0


def audit_and_validate_sum_costs(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
) -> Optional[Dict]:
    """
    before_tool_callback for accountant_agent, applied to sum_costs.

    Delivers three capabilities in a single pass:
      1. Audit trail   — logs every invocation with costs list and iteration number
      2. Validation    — silently removes negative or non-numeric values before tool runs
      3. High-cost flag — warns when any single cost exceeds HIGH_COST_THRESHOLD

    Return value contract:
      Always returns None — this callback never blocks execution.
      When invalid values are found, args['costs'] is mutated in-place.
      The tool then receives the cleaned argument dict automatically.
    """
    global _sum_costs_iteration

    # -- Defensive guard: only act on sum_costs ---------------------------
    if tool.name != "sum_costs":
        return None

    _sum_costs_iteration += 1
    agent_name = tool_context.agent_name
    costs = args.get("costs", [])

    # -- 1. Audit trail ---------------------------------------------------
    print(f"[AUDIT] sum_costs called | agent: {agent_name} | iteration: {_sum_costs_iteration}")
    print(f"[AUDIT] costs submitted: {costs}")

    # -- 2. Argument validation and sanitization --------------------------
    cleaned = []
    for value in costs:
        # Check: must be numeric
        if not isinstance(value, (int, float)):
            print(f"[VALIDATE] Removed invalid value: {value!r} (non-numeric)")
            continue
        # Check: must be non-negative
        if value < 0:
            print(f"[VALIDATE] Removed invalid value: {value} (negative)")
            continue
        cleaned.append(value)

    if len(cleaned) != len(costs):
        args["costs"] = cleaned   # mutate in-place — tool receives cleaned list
        print(f"[AUDIT] Sanitized costs: {cleaned}")

    # -- 3. High-cost flagging --------------------------------------------
    flagged = False
    for value in args.get("costs", []):
        if isinstance(value, (int, float)) and value > HIGH_COST_THRESHOLD:
            print(f"[FLAG] Suspiciously high cost detected: {value} (threshold: {HIGH_COST_THRESHOLD})")
            flagged = True

    if flagged:
        print(f"[AUDIT] Proceeding with flagged costs.")
    elif len(cleaned) == len(costs):
        # Only print this if no sanitization happened and no flags were raised
        print(f"[AUDIT] All values valid. Proceeding.")

    # Always return None — never block tool execution
    return None

## 🗝️ 8. [CARRIED OVER from 8.7] Cache Key Helper

Unchanged from Lecture 8.7.


In [ ]:
# ============================================================
# Lecture 8.7 — Cache Key Helper  [NEW]
# ============================================================

import json


def _make_cache_key(tool_name: str, args: Dict[str, Any]) -> str:
    """
    Build a deterministic, namespaced cache key from a tool name and its arguments.

    Format:  cache:<tool_name>:<json_serialised_args>

    Design decisions:
      - 'cache:' prefix namespaces all cache entries away from other state keys
        (e.g. 'budget', 'topic', 'current_plan').
      - tool_name is included so a future cache serving multiple tools never
        collides across tool types.
      - The full args dict serialised as JSON means two different queries to
        the same tool always produce different keys.

    Example outputs:
      cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
      cache:Google_Search_agent:{"query": "budget catering New York under 3000"}
    """
    args_str = json.dumps(args)
    return f"cache:{tool_name}:{args_str}"


# Quick sanity check — run this cell to confirm key generation is deterministic
key1 = _make_cache_key("Google_Search_agent", {"query": "affordable venue New York"})
key2 = _make_cache_key("Google_Search_agent", {"query": "affordable venue New York"})
print(f"Key 1 : {key1}")
print(f"Key 2 : {key2}")
print(f"Match : {key1 == key2}  ← must be True")

Key 1 : cache:Google_Search_agent:{"query": "affordable venue New York"}
Key 2 : cache:Google_Search_agent:{"query": "affordable venue New York"}
Match : True  ← must be True


## 📖 9. [CARRIED OVER from 8.7] Define the Cache Read Callback (`before_tool_callback`)

Unchanged from Lecture 8.7. `read_search_cache` is still attached to `cost_cutter_agent`.


In [ ]:
# ============================================================
# Lecture 8.7 — Cache Read: before_tool_callback  [NEW]
# Design Pattern: Caching (P4)
# ============================================================


def read_search_cache(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
) -> Optional[Dict]:
    """
    before_tool_callback for cost_cutter_agent, applied to google_search_tool.

    Implements the READ half of a read-through cache backed by session state.

    Execution flow:
      1. Generate a deterministic cache key from tool name + serialised args.
      2. Check tool_context.state for an existing entry under that key.
      3. Cache HIT  → log [CACHE HIT], return the cached dict.
                       The ADK framework uses the returned dict as the tool's
                       result — the tool function never executes.
                       write_search_cache never fires (tool was skipped).
      4. Cache MISS → log [CACHE MISS], return None.
                       The tool executes normally.
                       write_search_cache fires afterwards to store the result.

    Return value contract:
      - None         → cache miss; tool runs; write callback fires after
      - Dict         → cache hit;  tool skipped; write callback never fires

    Why tool_context.state as the cache store?
      Session state persists across all agents within a session and across
      all iterations of the LoopAgent. It is the natural shared memory in ADK.
      Using it as a cache store requires no external infrastructure — just
      a namespaced key convention to avoid collisions with other state entries.
    """
    # -- Defensive guard: only cache google_search_tool calls ---------------
    if tool.name != "Google_Search_agent":
        return None

    cache_key = _make_cache_key(tool.name, args)

    print(f"[CACHE] read_search_cache | tool: {tool.name}")
    print(f"[CACHE] Key: {cache_key}")

    # -- Cache lookup -------------------------------------------------------
    cached_result = tool_context.state.get(cache_key)

    if cached_result is not None:
        print(f"[CACHE HIT] Returning cached result. Tool execution skipped.")
        return cached_result   # non-None return skips the tool entirely

    print(f"[CACHE MISS] No cached result found. Proceeding with tool execution.")
    return None   # None return allows tool to execute normally

## 📝 10. [CARRIED OVER from 8.7] Define the Cache Write Callback (`after_tool_callback`)

Unchanged from Lecture 8.7. `write_search_cache` is still attached to `cost_cutter_agent`.


In [ ]:
# ============================================================
# Lecture 8.7 — Cache Write: after_tool_callback  [NEW]
# Design Pattern: Caching (P4)
# ============================================================


def write_search_cache(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
    tool_response: Dict,
) -> Optional[Dict]:
    """
    after_tool_callback for cost_cutter_agent, applied to google_search_tool.

    Implements the WRITE half of a read-through cache backed by session state.

    Execution flow:
      1. Defensive guard — only cache google_search_tool results.
      2. Regenerate the same cache key using the same helper function.
      3. Store tool_response in state under that key.
      4. Return None — the original tool_response passes through unchanged.

    This callback only fires when the tool actually ran (i.e. on a cache miss).
    On a cache hit, read_search_cache returned the cached result directly and
    the tool was skipped — so after_tool_callback never fires.
    This asymmetry is natural ADK framework behaviour, not special logic here.

    Return value contract:
      Always returns None — the original tool_response is used unchanged.
      We store it, we don't modify it.

    Why the same _make_cache_key call?
      The key must be identical to the one read_search_cache generated for the
      same (tool, args) pair. Using the shared helper guarantees this.
    """
    # -- Defensive guard: only cache google_search_tool results -------------
    if tool.name != "Google_Search_agent":
        return None

    cache_key = _make_cache_key(tool.name, args)

    print(f"[CACHE] write_search_cache | storing result under key:")
    print(f"[CACHE] Key: {cache_key}")

    # -- Store result in session state -------------------------------------
    tool_context.state[cache_key] = tool_response
    print(f"[CACHE WRITE] Result cached successfully.")

    # Return None — original tool_response passes through to the LLM unchanged
    return None

## 📊 11. [NEW] Define the Budget History Callback (`after_tool_callback` on `sum_costs`)

`track_budget_history` is the **write side** of the state bridge — it fires after every `sum_costs` execution inside the loop and appends the running total to `state['budget_history']`.

### Callback signature

```python
def track_budget_history(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
    tool_response: Dict        # ← the result sum_costs just produced
) -> Optional[Dict]:
```

### What it does

1. **Defensive guard** — checks `tool.name == 'sum_costs'`; returns `None` immediately for any other tool on `accountant_agent`
2. **Extracts the total** — reads the running cost from `tool_response`
3. **List accumulation** — reads the existing list from state, appends, reassigns (never in-place mutation)
4. **Always returns `None`** — this callback is pure state writing; it never blocks or modifies the tool result

### State accumulation pattern

```python
# ✅ CORRECT — read → append → reassign
history = tool_context.state.get('budget_history', [])
history.append(total)
tool_context.state['budget_history'] = history

# ❌ WRONG — in-place mutation; state proxy may not propagate
tool_context.state['budget_history'].append(total)
```

### Return value contract

| Return value | What happens |
|---|---|
| `None` (always) | Original `tool_response` passes through unchanged to the LLM |

This callback is **pure state writing** — a third distinct use case for `after_tool_callback` after blocking (8.4/8.7) and caching (8.7).


In [ ]:
# ============================================================
# Lecture 8.8 — Budget History Tracker: after_tool_callback  [NEW]
# Design Pattern: Dynamic State Management (P2)
# ============================================================


def track_budget_history(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
    tool_response: Dict,
) -> Optional[Dict]:
    """
    after_tool_callback for accountant_agent, applied to sum_costs.

    Implements the WRITE side of the dynamic state bridge:
      - Extracts the running cost total from the tool result
      - Appends it to state['budget_history'] — a growing list, one entry per loop iteration
      - Always returns None — the original tool_response passes through unchanged

    By the time the loop terminates, budget_history contains one entry per
    iteration: the exact total the accountant computed each time.

    State accumulation pattern (CRITICAL):
      # ✅ CORRECT — read → append → reassign
      history = tool_context.state.get('budget_history', [])
      history.append(total)
      tool_context.state['budget_history'] = history

      # ❌ WRONG — in-place mutation on a nested object
      tool_context.state['budget_history'].append(total)
      # The state proxy may not propagate in-place mutations on nested objects.
      # Always reassign.

    Return value contract:
      Always returns None — never blocks or modifies the tool result.
      This callback is pure state writing.
    """
    # -- Defensive guard: only act on sum_costs ---------------------------


    # -- Extract the total from the tool result ---------------------------
    # sum_costs returns a float directly; tool_response wraps it.
    # The framework stores the raw return value of the tool function.


    # -- List accumulation (read → append → reassign) ---------------------


    print(f"[HISTORY] Appended {float(total):,.1f} to budget_history → {history}")

    # Always return None — original tool_response used unchanged
    return None


## 🎯 12. [NEW] Define the Enrichment Callback (`before_agent_callback` on `plan_retriever_agent`)

`enrich_final_output` is the **read side** of the state bridge — it fires once, right before `plan_retriever_agent` starts, after all loop iterations have completed.

### Callback signature

```python
def enrich_final_output(
    callback_context: CallbackContext,
) -> Optional[types.Content]:
```

### What it does

1. **Reads `state['budget_history']`** — the list populated by `track_budget_history` across all loop iterations
2. **Formats a refinement summary** — iteration count, starting total, final total, savings
3. **Writes `state['refinement_summary']`** — so `plan_retriever_agent` can access it via its `{{refinement_summary}}` template variable
4. **Always returns `None`** — the agent proceeds normally with enriched state; no skipping or replacement

### The state bridge in one picture

```
track_budget_history         state['budget_history']        enrich_final_output
(after_tool on sum_costs)  ─────── WRITES ──────────►  ──── READS ────────────►
                                                          writes refinement_summary
                                                          → plan_retriever_agent
                                                            reads {{refinement_summary}}
```

The two callbacks never call each other. State is the only coupling.

### Return value contract

| Condition | Return value | What happens |
|---|---|---|
| `budget_history` missing or empty | `None` | Agent runs normally, no enrichment |
| `budget_history` exists | `None` | State enriched; agent runs normally with `{{refinement_summary}}` available |

This callback **never** returns a `Content` object — it only enriches state. The agent always runs.


In [ ]:
# ============================================================
# Lecture 8.8 — Final Output Enrichment: before_agent_callback  [NEW]
# Design Pattern: Dynamic State Management (P2)
# ============================================================


def enrich_final_output(
    callback_context: CallbackContext,
) -> Optional[types.Content]:
    """
    before_agent_callback for plan_retriever_agent.

    Implements the READ side of the dynamic state bridge:
      - Reads state['budget_history'] — the list accumulated by track_budget_history
      - Formats a human-readable refinement summary
      - Writes the summary into state['refinement_summary']
      - Returns None — the agent proceeds normally with the enriched state

    plan_retriever_agent's instruction contains {{refinement_summary}}, which the
    ADK framework will resolve from state when the agent renders its prompt.

    This is the same before_agent_callback hook used in 8.3 for logging —
    but here it serves a completely different purpose: state enrichment.

    Return value contract:
      Always returns None — never skips or replaces the agent's output.
      Enriching state is sufficient; there is no need to intercept the agent.

    Guard behaviour:
      If budget_history is missing or empty, returns None immediately.
      The agent runs with no enrichment — graceful degradation.
    """
    agent_name = callback_context.agent_name
    print(f"[ENRICH] {agent_name} starting")

    # -- Read the budget history written by track_budget_history -----------


    print(f"[ENRICH] budget_history found: {history}")

    # -- Format the refinement summary ------------------------------------


    summary = (
        f"Refined across {iteration_count} iteration{'s' if iteration_count != 1 else ''}. "
        f"Total reduced from ${starting_total:,.0f} → ${final_total:,.0f} "
        f"(saving ${savings:,.0f})."    )

    print(f"[ENRICH] Wrote refinement_summary to state:")
    print(f"         \"{summary}\"")
    print(f"[ENRICH] Returning None — agent proceeds normally with enriched state.")

    # -- Write the summary into state -------------------------------------


    # Always return None — the agent runs normally, reads {{refinement_summary}}



## 🛠️ 13. Define Workflow Tools

**Unchanged from Lecture 8.7.** The dynamic state callbacks are attached to the agents, not the tool definitions.


In [ ]:
from google.adk.tools import ToolContext

def sum_costs(costs: list[float]) -> float:
    """Calculates the sum of a list of numbers."""
    print(f"  [Tool Call] sum_costs on the list: {costs}")
    return sum(costs)

def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the plan is approved and within budget."""
    print(f"  [Tool Call] Budget approved. Terminating loop: {json.dumps(tool_context.state.to_dict())}")
    tool_context.actions.escalate = True
    return None

## 14. Create Tool Wrappers

Unchanged from Lecture 8.7.


In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_agent",
    model=AGENT_MODEL,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)

google_search_tool = AgentTool(agent=google_search_agent)

## 📝 15. Create Agents

**Three changes from Lecture 8.7** — highlighted with `# <- 8.8 CHANGE`:

- `accountant_agent` — gains `after_tool_callback=track_budget_history` (joins the existing `before_tool_callback=audit_and_validate_sum_costs`)
- `plan_retriever_agent` — gains `before_agent_callback=enrich_final_output` (joins the existing `after_agent_callback=log_agent_exit`); instruction updated to include `{{refinement_summary}}`
- `spending_proposer_agent`, `cost_cutter_agent` — unchanged from 8.7

All nine callbacks now fire simultaneously across the workflow:

| Agent | Callback | Hook | Pattern |
|---|---|---|---|
| `budget_optimizer_workflow` | `guardrail_before_workflow` | `before_agent_callback` | P1 Guardrails |
| `spending_proposer_agent` | `log_agent_entry` | `before_agent_callback` | P3 Logging |
| `cost_cutter_agent` | `sanitize_cost_cutter_response` | `after_model_callback` | P5 Modification |
| `accountant_agent` | `audit_and_validate_sum_costs` | `before_tool_callback` | P5 Tool-layer |
| `accountant_agent` | `track_budget_history` | `after_tool_callback` | P2 Dynamic State ← **8.8** |
| `cost_cutter_agent` | `read_search_cache` | `before_tool_callback` | P4 Caching |
| `cost_cutter_agent` | `write_search_cache` | `after_tool_callback` | P4 Caching |
| `plan_retriever_agent` | `enrich_final_output` | `before_agent_callback` | P2 Dynamic State ← **8.8** |
| `plan_retriever_agent` | `log_agent_exit` | `after_agent_callback` | P3 Logging |


In [ ]:
COMPLETION_PHRASE = "The plan is within the budget."

# Agent 1: Proposes the initial, expensive plan (runs once).
# before_agent_callback=log_agent_entry carried over from 8.3 unchanged.
spending_proposer_agent = Agent(
    name="spending_proposer_agent",
    model=AGENT_MODEL,
    tools=[google_search],
    instruction="""
    You are a luxury event planner. For a {{topic}}, find a high-end venue and a gourmet catering service.

    Output a JSON object with items and their estimated costs, like:
    {"venue": {"name": "The Ritz London", "cost": 10000}, "catering": {"name": "Gourmet Chefs Inc.", "cost": 5000}}
    """,
    output_key="current_plan",
    before_agent_callback=log_agent_entry,   # <- 8.3/8.4/8.5/8.6/8.7 (unchanged)
)

# Agent 2 (in loop): The "Accountant" that critiques the plan.
# before_tool_callback added in 8.6 — unchanged.
# after_tool_callback=track_budget_history added in 8.8.
accountant_agent = Agent(
    name="accountant_agent",
    model=AGENT_MODEL,
    tools=[sum_costs],
    before_tool_callback=audit_and_validate_sum_costs,   # <- 8.6 (unchanged)
    # <- 8.8 CHANGE
    instruction=f"""
    You are a meticulous accountant. Your budget is {{{{budget}}}}.
    The current plan is: {{{{current_plan}}}}

    Extract the costs from the plan and use the `sum_costs` tool to get the total.
    - IF the total cost is > {{{{budget}}}}, output a critique like: "This plan is over budget by [amount]. Find a cheaper [item]."
    - ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key="critique",
)

# Agent 3 (in loop): The "Cost Cutter" that refines the plan.
# after_model_callback from 8.5 — unchanged.
# before_tool_callback + after_tool_callback from 8.7 — unchanged.
cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=AGENT_MODEL,
    tools=[google_search_tool, exit_loop],
    instruction=f"""
    You are a cost-cutting expert. You must refine a plan based on a critique.
    The critique is: {{{{critique}}}}
    The current plan is: {{{{current_plan}}}}

    - IF the critique is '{COMPLETION_PHRASE}'
        1. You MUST call the `exit_loop` tool with no arguments.
        2. After calling exit_loop, output the current plan EXACTLY as-is, character for character,
           with no modifications, no acknowledgements, no commentary, and no extra text. Do not summarize it.
           Do not rephrase it. Do not add "Budget approved" or any other text.
           Just echo {{{{current_plan}}}} verbatim.
    - ELSE, read the critique to identify the overpriced item. Use your search tool to find a cheaper alternative for that item.
      Output a new JSON object with the updated plan.
    """,
    output_key="current_plan",
    after_model_callback=sanitize_cost_cutter_response,   # <- 8.5 (unchanged)
    before_tool_callback=read_search_cache,               # <- 8.7 (unchanged)
    after_tool_callback=write_search_cache,               # <- 8.7 (unchanged)
)

# Agent 4: Presents the final approved plan (runs once after loop).
# after_agent_callback=log_agent_exit carried over from 8.3.
# before_agent_callback=enrich_final_output added in 8.8.
# Instruction updated in 8.8 to include {{refinement_summary}} — the only instruction
# change in the entire Section 8.
plan_retriever_agent = Agent(
    name="plan_retriever_agent",
    model=AGENT_MODEL,
    instruction="""
    You are a plan finalizer. Your only job is to present the final, approved plan.
    The plan is available in the context variable `{{current_plan}}`.

    If available, also include the following refinement summary at the end of your output:
    {{refinement_summary}}

    Your output must be the content of the final plan presented in a clear and
    easy-to-read format.
    """,
    tools=[],
    output_key="final_presentation",
    # <- 8.8 CHANGE
    after_agent_callback=log_agent_exit,          # <- 8.3/8.4/8.5/8.6/8.7 (unchanged)
)


## 🔄 16. Assemble the Loop and Sequential Workflows

Unchanged from Lecture 8.7. The dynamic state callbacks live on the agent definitions — no changes needed to the LoopAgent or SequentialAgent definitions.


In [ ]:
from google.adk.agents import SequentialAgent, LoopAgent

# Unchanged from Section 5.
budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3,
)

# <- 8.4 (unchanged): guardrail still lives on the SequentialAgent.
# 8.7 changes are on cost_cutter_agent, not here.
budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[spending_proposer_agent, budget_refinement_loop, plan_retriever_agent],
    before_agent_callback=guardrail_before_workflow,   # <- 8.4 (unchanged)
)

## 🚀 17. Build the Execution Engine

Unchanged from Lecture 8.7. All nine callbacks fire automatically inside the runner — the runner does not need to know about them.


In [ ]:
from IPython.display import display, Markdown

from google.adk.sessions import Session
from google.genai.types import Content, Part
from google.adk.runners import Runner

async def run_agent_query(agent: Agent, query: str, topic: str, budget: str, session: Session, user_id: str):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
            state_delta={"budget": budget, "topic": topic, "COMPLETION_PHRASE": COMPLETION_PHRASE}
        ):
            pass
    except Exception as e:
        final_response = f"An error occurred: {e}"
        return final_response

    # Read the final response from session state.
    #
    # Two possible paths through the workflow:
    #
    #   ✅ CLEAN topic  → plan_retriever_agent runs and writes final_presentation.
    #                     We read that.
    #
    #   🚫 BLOCKED topic → guardrail cancels the workflow and writes refusal_reason
    #                      into state. plan_retriever never runs, so
    #                      final_presentation is never written. We read
    #                      refusal_reason instead.
    #
    final_session = await session_service.get_session(
        app_name=agent.name,
        user_id=user_id,
        session_id=session.id
    )
    state = final_session.state

    if "final_presentation" in state:
        final_response = state["final_presentation"]
    elif "refusal_reason" in state:
        final_response = state["refusal_reason"]
    else:
        final_response = "No response was generated."

    print("\n" + "-"*50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-"*50 + "\n")

    return final_response

## ✨ 18. Initialize Session Service

Unchanged from Lecture 8.7.


In [ ]:
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
user_id = "adk_event_planner_001"

## 🔬 19. [CARRIED OVER from 8.7] Standalone Cache Demo

The four-scenario cache test from Lecture 8.7 — carried forward unchanged.

This cell defines the **shared mock classes** used by both this demo and the Dynamic State demo below:
- `_MockState` — real dict subclass used as `tool_context.state`
- `_MockTool` — minimal stand-in for `BaseTool`, only needs `.name`
- `_MockToolContext` — minimal stand-in for `ToolContext`, backed by `_MockState`
- `make_mock_tool(name)` / `make_mock_tool_context()` — factory helpers

**Why not use `MagicMock`?** Python resolves dunder methods (`__setitem__`) on the *class*, not the instance. Assigning a lambda to `mock.__setitem__` is silently ignored and `self` gets injected as an extra argument — causing a `TypeError` the moment `write_search_cache` tries to store a result. A plain dict subclass avoids this entirely.


In [ ]:
# ============================================================
# Lecture 8.7 — Standalone Cache Demo  [CARRIED OVER — UNCHANGED]
# ============================================================

# ── Shared mock infrastructure ────────────────────────────────────────────────
# These classes are used by BOTH this cache demo and the Dynamic State demo below.
# Define them here once so both demo cells can reference them.

class _MockState(dict):
    """A real dict that also supports .get() — used as ctx.state."""
    pass

class _MockTool:
    """Minimal stand-in for BaseTool — only the name attribute is needed."""
    def __init__(self, name):
        self.name = name

class _MockToolContext:
    """
    Minimal stand-in for ToolContext.

    Uses a real dict for state so that:
        tool_context.state.get(key)      — works correctly (returns None on miss)
        tool_context.state[key] = value  — works correctly (stores the value)

    MagicMock cannot be used here: Python looks up dunder methods (__setitem__)
    on the *class*, not the instance, so assigning a lambda to the instance
    attribute is silently ignored and self gets injected as an extra argument.
    A plain dict subclass avoids the problem entirely.
    """
    def __init__(self, agent_name="cost_cutter_agent"):
        self.agent_name = agent_name
        self.state = _MockState()

def make_mock_tool(name):
    """Build a minimal mock BaseTool with just a name attribute."""
    return _MockTool(name)

def make_mock_tool_context():
    """Build a minimal mock ToolContext backed by a real dict."""
    return _MockToolContext()

mock_tool = make_mock_tool("Google_Search_agent")
mock_ctx  = make_mock_tool_context()

FAKE_RESULT   = {"results": [{"title": "Affordable Venue NYC",  "url": "https://example.com"}]}
FAKE_RESULT_2 = {"results": [{"title": "Budget Caterer NYC",    "url": "https://caterer.com"}]}

# ── Scenario 1 — First call: cache miss, write fires ──────────────────────────
print("=" * 65)
print("SCENARIO 1 — First call (cache miss → tool runs → write fires)")
print("=" * 65)
args1 = {"query": "affordable venue New York 50 people"}

result = read_search_cache(mock_tool, args1, mock_ctx)
print(f"→ read_search_cache returned: {result}  (None = proceed with tool)")

print("  [Tool Call] google_search_tool executes... (simulated ~2-3 seconds)")

write_search_cache(mock_tool, args1, mock_ctx, FAKE_RESULT)
print(f"→ write_search_cache called: result stored in state")

# ── Scenario 2 — Same query: cache hit, tool skipped, write never fires ───────
print()
print("=" * 65)
print("SCENARIO 2 — Repeated call, same query (cache hit → tool skipped)")
print("=" * 65)
args2 = {"query": "affordable venue New York 50 people"}   # identical to args1

result = read_search_cache(mock_tool, args2, mock_ctx)
print(f"→ read_search_cache returned: {result}")
print(f"  (Non-None return = tool skipped entirely; write_search_cache never fires)")

# ── Scenario 3 — Different query: cache miss with a new key ───────────────────
print()
print("=" * 65)
print("SCENARIO 3 — Different query (cache miss → new key → write fires)")
print("=" * 65)
args3 = {"query": "budget catering New York under 3000"}   # different from args1

result = read_search_cache(mock_tool, args3, mock_ctx)
print(f"→ read_search_cache returned: {result}  (None = proceed with tool)")

print("  [Tool Call] google_search_tool executes... (simulated)")
write_search_cache(mock_tool, args3, mock_ctx, FAKE_RESULT_2)
print(f"→ write_search_cache called: second key stored")

# ── Scenario 4 — Loop simulation: misses in iter 1, hits in iters 2 & 3 ──────
print()
print("=" * 65)
print("SCENARIO 4 — Loop simulation: hits accumulate across iterations")
print("=" * 65)

loop_ctx       = make_mock_tool_context()
venue_query    = {"query": "cheaper venue New York"}
catering_query = {"query": "affordable catering NYC"}
dj_query       = {"query": "budget DJ New York"}

for iteration in range(1, 4):
    print(f"\n=== Loop Iteration {iteration} ===")

    # Venue query — miss in iteration 1, hit in 2 & 3
    r = read_search_cache(mock_tool, venue_query, loop_ctx)
    if r is None:
        print("  → venue: tool runs")
        write_search_cache(mock_tool, venue_query, loop_ctx, {"results": ["Venue result"]})
    else:
        print("  → venue: instant cache return (no API call)")

    # Catering query — asked once in iteration 1 only
    if iteration == 1:
        r = read_search_cache(mock_tool, catering_query, loop_ctx)
        if r is None:
            print("  → catering: tool runs")
            write_search_cache(mock_tool, catering_query, loop_ctx, {"results": ["Catering result"]})

    # DJ query — miss in iteration 2, hit in iteration 3
    if iteration >= 2:
        r = read_search_cache(mock_tool, dj_query, loop_ctx)
        if r is None:
            print("  → DJ: tool runs")
            write_search_cache(mock_tool, dj_query, loop_ctx, {"results": ["DJ result"]})
        else:
            print("  → DJ: instant cache return (no API call)")


SCENARIO 1 — First call (cache miss → tool runs → write fires)
[CACHE] read_search_cache | tool: Google_Search_agent
[CACHE] Key: cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
[CACHE MISS] No cached result found. Proceeding with tool execution.
→ read_search_cache returned: None  (None = proceed with tool)
  [Tool Call] google_search_tool executes... (simulated ~2-3 seconds)
[CACHE] write_search_cache | storing result under key:
[CACHE] Key: cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
[CACHE WRITE] Result cached successfully.
→ write_search_cache called: result stored in state

SCENARIO 2 — Repeated call, same query (cache hit → tool skipped)
[CACHE] read_search_cache | tool: Google_Search_agent
[CACHE] Key: cache:Google_Search_agent:{"query": "affordable venue New York 50 people"}
[CACHE HIT] Returning cached result. Tool execution skipped.
→ read_search_cache returned: {'results': [{'title': 'Affordable Venue NYC', 'url': 'h

## 🔬 20. [NEW] Standalone Dynamic State Demo

Verify both new callbacks in complete isolation — no runner, no session, no LLM required.
The mock classes (`_MockState`, `_MockTool`, `_MockToolContext`, `make_mock_tool`, `make_mock_tool_context`)
are defined in the cache demo cell above and reused here.

This cell demonstrates **two scenarios**:

**Scenario 1** — `track_budget_history` accumulating across three loop iterations
**Scenario 2** — `enrich_final_output` reading the history and writing the refinement summary


In [ ]:
# ============================================================
# Standalone demo — verify dynamic state callbacks with mock objects
# ============================================================
# Mock classes (_MockState, _MockTool, _MockToolContext, make_mock_tool,
# make_mock_tool_context) are defined in the cache demo cell above.

# ── Shared mock setup ─────────────────────────────────────────────────────────
mock_sum_costs_tool = make_mock_tool("sum_costs")
mock_wrong_tool     = make_mock_tool("exit_loop")
mock_acc_ctx        = make_mock_tool_context()
mock_acc_ctx.agent_name = "accountant_agent"

# ── Scenario 1 — track_budget_history accumulating across three iterations ────
print("=" * 65)
print("SCENARIO 1 — track_budget_history accumulating across loop iterations")
print("=" * 65)

iteration_costs = [18000.0, 15500.0, 12400.0]

for i, total in enumerate(iteration_costs, 1):
    print(f"\n=== Loop Iteration {i} ===")
    result = track_budget_history(mock_sum_costs_tool, {"costs": []}, mock_acc_ctx, total)
    print(f"→ track_budget_history returned: {result}  (None = tool result unchanged)")

print(f"\n→ Final budget_history in state: {mock_acc_ctx.state.get('budget_history')}")

# Confirm wrong tool is guarded
print("\n--- Defensive guard: wrong tool ---")
guard_result = track_budget_history(mock_wrong_tool, {}, mock_acc_ctx, 999.0)
print(f"→ exit_loop tool: returned {guard_result}  (None = immediately returned, no append)")
print(f"→ budget_history unchanged: {mock_acc_ctx.state.get('budget_history')}")

# ── Scenario 2 — enrich_final_output reading history and writing summary ──────
print()
print("=" * 65)
print("SCENARIO 2 — enrich_final_output reading history and writing summary")
print("=" * 65)

class _MockCallbackContext:
    """Minimal stand-in for CallbackContext — used by enrich_final_output."""
    def __init__(self, state_dict):
        self.agent_name = "plan_retriever_agent"
        self.invocation_id = "mock-invocation-001"
        self.state = _MockState(state_dict)

mock_cb_ctx = _MockCallbackContext(mock_acc_ctx.state)

print()
result = enrich_final_output(mock_cb_ctx)
print(f"\n→ enrich_final_output returned: {result}  (None = agent proceeds normally)")
print(f"→ refinement_summary in state:")
print(f"   '{mock_cb_ctx.state.get('refinement_summary')}'")

# Confirm empty history is handled gracefully
print()
print("--- Guard: empty budget_history ---")
empty_ctx = _MockCallbackContext({})
result = enrich_final_output(empty_ctx)
print(f"→ empty history: returned {result}  (None = no enrichment, agent runs normally)")


SCENARIO 1 — track_budget_history accumulating across loop iterations

=== Loop Iteration 1 ===
[HISTORY] Appended 18,000.0 to budget_history → [18000.0]
→ track_budget_history returned: None  (None = tool result unchanged)

=== Loop Iteration 2 ===
[HISTORY] Appended 15,500.0 to budget_history → [18000.0, 15500.0]
→ track_budget_history returned: None  (None = tool result unchanged)

=== Loop Iteration 3 ===
[HISTORY] Appended 12,400.0 to budget_history → [18000.0, 15500.0, 12400.0]
→ track_budget_history returned: None  (None = tool result unchanged)

→ Final budget_history in state: [18000.0, 15500.0, 12400.0]

--- Defensive guard: wrong tool ---
→ exit_loop tool: returned None  (None = immediately returned, no append)
→ budget_history unchanged: [18000.0, 15500.0, 12400.0]

SCENARIO 2 — enrich_final_output reading history and writing summary

[ENRICH] plan_retriever_agent starting
[ENRICH] budget_history found: [18000.0, 15500.0, 12400.0]
[ENRICH] Wrote refinement_summary to state:

## ▶️ 21a. Run — CLEAN Topic (Capstone Workflow Run)

The topic `"50 person AI event in New York"` passes the guardrail and runs the full nine-callback workflow.

This is the **capstone run** for Section 8 — all nine callbacks firing simultaneously across the workflow for the first time.

Watch for:
- **`[HISTORY]`** lines after every `sum_costs` call — showing `budget_history` growing with each iteration
- **`[ENRICH]`** lines right before `plan_retriever_agent` starts — showing the summary being constructed
- **The final output** — which now includes the refinement summary at the bottom: how many iterations ran, what the starting total was, what the final total is, and how much was saved

You will also see all prior callback layers active simultaneously:
- `[SAFETY JUDGE]` — guardrail evaluates the topic
- `[ENTRY]` — `log_agent_entry` fires as `spending_proposer_agent` starts
- `[SANITIZE]` — `sanitize_cost_cutter_response` fires after each `cost_cutter_agent` LLM call
- `[AUDIT]` — `audit_and_validate_sum_costs` fires before each `sum_costs` tool call
- `[HISTORY]` — `track_budget_history` fires after each `sum_costs` tool call ← **NEW**
- `[CACHE]` — `read_search_cache` + `write_search_cache` fire on each `google_search_tool` call
- `[ENRICH]` — `enrich_final_output` fires before `plan_retriever_agent` starts ← **NEW**
- `[EXIT]` — `log_agent_exit` fires after `plan_retriever_agent` completes

**Nine callbacks. Four agents. Six patterns. One workflow — unchanged in structure since Section 5.**


In [ ]:
import time

# Reset the auditing iteration counter before the live run
_sum_costs_iteration = 0

async def run_clean_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "50 person AI event in New York"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")

    t_start = time.time()
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)
    t_end = time.time()

    print(f"\n⏱️  Total workflow time: {t_end - t_start:.1f}s")
    print("\n--- Capstone callback map fired in this run ---")
    print("  [SAFETY JUDGE] → guardrail_before_workflow      (P1 Guardrails)")
    print("  [ENTRY]        → log_agent_entry                (P3 Logging)")
    print("  [SANITIZE]     → sanitize_cost_cutter_response  (P5 Modification)")
    print("  [AUDIT]        → audit_and_validate_sum_costs   (P5 Tool-layer)")
    print("  [HISTORY]      → track_budget_history           (P2 Dynamic State) ← NEW")
    print("  [CACHE]        → read_search_cache              (P4 Caching)")
    print("  [CACHE]        → write_search_cache             (P4 Caching)")
    print("  [ENRICH]       → enrich_final_output            (P2 Dynamic State) ← NEW")
    print("  [EXIT]         → log_agent_exit                 (P3 Logging)")

await run_clean_topic()


User: Find a plan for 50 person AI event in New York


🚀 Running query for agent: 'budget_optimizer_workflow' in session: '314891be-b81d-4bc9-926a-6d20449cb153'...

────────────────────────────────────────────────────────────
[SAFETY JUDGE] Evaluating topic: '50 person AI event in New York'
[SAFETY JUDGE] Verdict         : NO
  └─ ✅ SAFE — starting workflow.
────────────────────────────────────────────────────────────

[ENTRY] spending_proposer_agent
        inv        : e-574e9018-dd8e-43ff-8dab-649ccc71d5f4
        timestamp  : 23:26:19
        state_keys : ['budget', 'topic', 'COMPLETION_PHRASE']


[AUDIT] sum_costs called | agent: accountant_agent | iteration: 1
[AUDIT] costs submitted: [10000, 12500]
[FLAG] Suspiciously high cost detected: 12500 (threshold: 10000)
[AUDIT] Proceeding with flagged costs.
  [Tool Call] sum_costs on the list: [10000, 12500]
[HISTORY] Appended 22,500.0 to budget_history → [22500.0]
[CACHE] read_search_cache | tool: Google_Search_agent
[CACHE] Key: cache:Google_Search_agent:{"request": "cheaper venues for 50-person AI event New York"}
[CACHE MISS] No cached result found. Proceeding with tool execution.
[CACHE] write_search_cache | storing result under key:
[CACHE] Key: cache:Google_Search_agent:{"request": "cheaper venues for 50-person AI event New York"}
[CACHE WRITE] Result cached successfully.
[SANITIZE] Response already clean. Passing through.
[AUDIT] sum_costs called | agent: accountant_agent | iteration: 2
[AUDIT] costs submitted: [2500, 12500]
[FLAG] Suspiciously high cost detected: 12500 (threshold: 10000)
[AUDIT] Proceeding with flagged cost

For a high-end, 50-person AI event in New York, the following luxury venue and gourmet catering service are proposed:

### Venue: The Farm Soho Loft

**Description:** The Farm Soho Loft in SoHo offers a versatile and technologically equipped environment suitable for an AI event. This 1,000-square-foot historic loft features 15-foot ceilings, a built-in AV system, a projector, and a sound system. It can comfortably host 50 guests in a private setting and is ideal for corporate off-sites and high-impact presentations, with hi-tech meeting rooms and expert tech support. The venue allows outside catering, providing flexibility for cost management.

**Estimated Cost:** Event venues at The Farm SoHo start from $500+/hour. For a comprehensive event for 50 guests, including the use of their AV system and tech support for approximately 5 hours, an estimated cost of **$2,500** is appropriate. This offers a significant cost saving while maintaining a professional and technologically capable environment for the AI event.

### Catering: Relish Catering + Hospitality

**Description:** Relish Catering + Hospitality is a premier, full-service catering company in New York City, renowned for "providing full-service catering for corporate events, weddings, private parties, and workplace dining". Their team combines culinary expertise with professional event management, offering "globally inspired, farm-to-fork menus tailored to your event". They are known for creating "unforgettable experiences" and handle everything from menu planning to setup and cleanup.

**Estimated Cost:** For high-end corporate catering in New York City, particularly for a full-service plated event with gourmet offerings, costs can range from $75 to $150 or more per person. Relish Events, associated with Relish Catering + Hospitality, indicates food starting at $150 per person. For a luxury event for 50 guests, including a gourmet menu, beverages, professional staffing, and necessary rentals (linens, dishware, etc.), a high-end estimate of approximately $250 per person is suitable. Therefore, the estimated cost for catering would be **$12,500**.

---

Refined across 2 iterations. Total reduced from $22,500 → $15,000 (saving $7,500).

--------------------------------------------------


⏱️  Total workflow time: 49.6s

--- Capstone callback map fired in this run ---
  [SAFETY JUDGE] → guardrail_before_workflow      (P1 Guardrails)
  [ENTRY]        → log_agent_entry                (P3 Logging)
  [SANITIZE]     → sanitize_cost_cutter_response  (P5 Modification)
  [AUDIT]        → audit_and_validate_sum_costs   (P5 Tool-layer)
  [HISTORY]      → track_budget_history           (P2 Dynamic State) ← NEW
  [CACHE]        → read_search_cache              (P4 Caching)
  [CACHE]        → write_search_cache             (P4 Caching)
  [ENRICH]       → enrich_final_output            (P2 Dynamic State) ← NEW
  [EXIT]         → log_agent_exit                 (P3 Logging)


## 🚫 21b. Run — BLOCKED Topic (Guardrail Still Intercepts)

The topic `"weapons convention"` is still blocked by the 8.4 guardrail.

Notice: `[HISTORY]` and `[ENRICH]` lines never appear — because `accountant_agent` and `plan_retriever_agent` never run when the workflow is cancelled at the outermost boundary. This confirms the new callbacks only fire when their respective agents and tools are actually invoked.


In [ ]:
async def run_blocked_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "weapons convention"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_blocked_topic()

User: Find a plan for weapons convention


🚀 Running query for agent: 'budget_optimizer_workflow' in session: '6d1b26ad-1979-4e85-8baa-8d5a9fde01a1'...

────────────────────────────────────────────────────────────
[SAFETY JUDGE] Evaluating topic: 'weapons convention'
[SAFETY JUDGE] Verdict         : YES
[SAFETY JUDGE] Generating refusal explanation...
[SAFETY JUDGE] Reason          : Dear [Client Name],

Thank you for your inquiry regarding your event. While we appreciate your interest, we must respectfully decline to assist with planning a 'weapons convention,' as the nature of such an event presents inherent safety and liability concerns that are outside the scope of our services and company policy. We wish you the best in your planning.
  └─ 🚫 BLOCKED — high-risk topic. Workflow cancelled.
────────────────────────────────────────────────────────────

--------------------------------------------------
✅ Final Response:


Dear [Client Name],

Thank you for your inquiry regarding your event. While we appreciate your interest, we must respectfully decline to assist with planning a 'weapons convention,' as the nature of such an event presents inherent safety and liability concerns that are outside the scope of our services and company policy. We wish you the best in your planning.

--------------------------------------------------

